# User 1, movie 20, held out on purpose

The first row of `ua.test` is user 1, movie 20, rating 4. Movie 20 is *Angels and Insects (1995)*. This rating is one of user 1's ten test ratings, so it is not in `ua.base`.

In the base file, user 1 has 262 ratings summing to 944.

$$
\bar r_{u} = \dfrac{944}{262} = \dfrac{472}{131}.
$$

The other ten ratings, all in the test file, sum to 38. The full-file sum was 982, and $944 + 38 = 982$.

Movie 20, inside the base file, has 61 ratings summing to 204.

$$
\bar r_{i} = \dfrac{204}{61}.
$$

The base mean is $\mu = 319153/90570$. The bias prediction is

$$
\dfrac{472}{131} + \dfrac{204}{61} - \dfrac{319153}{90570}
= \dfrac{2477732497}{723744870}.
$$

That is about 3.423. The hidden rating is 4. The miss is about 0.577 stars, on this one row. The RMSE of about 0.992 is the root mean square of 9430 such misses, not this one.

User 1's ten hidden ratings are the movies 20, 33, 61, 117, 155, 160, 171, 189, 202, and 265, with stars 4, 4, 4, 3, 2, 4, 5, 3, 5, and 4. Any mean built for scoring those ten has to leave all ten out.


In [2]:
# Locate the MovieLens tables beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "16-Movie Recommendation" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "u.info").exists():
            return direct
    raise FileNotFoundError(name)

def read_lines(name):
    return data_path(name).read_text(encoding="latin-1").splitlines()

def read_ratings(name):
    rows = []
    for line in read_lines(name):
        if not line.strip():
            continue
        user, item, rating, stamp = line.split("\t")
        rows.append((int(user), int(item), int(rating), int(stamp)))
    return rows

ratings = read_ratings("u.data")
print(len(ratings), ratings[0])


100000 (196, 242, 3, 881250949)


In [3]:
# The held-out Angels and Insects rating, as an exact fraction.
from fractions import Fraction
from collections import defaultdict
movies = {int(row[0]): row[1] for row in (line.split("|") for line in read_lines("u.item"))}
base = read_ratings("ua.base")
test = read_ratings("ua.test")
assert test[0] == (1, 20, 4, 887431883)
assert movies[20] == "Angels and Insects (1995)"
mu = Fraction(sum(rating for _, _, rating, _ in base), len(base))
user_ratings = [rating for user, _, rating, _ in base if user == 1]
movie_ratings = [rating for _, item, rating, _ in base if item == 20]
user_mean = Fraction(sum(user_ratings), len(user_ratings))
movie_mean = Fraction(sum(movie_ratings), len(movie_ratings))
prediction = user_mean + movie_mean - mu
hidden = [(item, rating) for user, item, rating, _ in test if user == 1]
print(user_mean, movie_mean, prediction, float(prediction))
print(hidden)
assert user_mean == Fraction(472, 131)
assert movie_mean == Fraction(204, 61)
assert mu == Fraction(319153, 90570)
assert prediction == Fraction(2477732497, 723744870)
assert abs(float(prediction) - 3.4234888559555525) < 1e-12
assert sum(rating for _, rating in hidden) == 38
assert 944 + 38 == 982
assert [item for item, _ in hidden] == [20, 33, 61, 117, 155, 160, 171, 189, 202, 265]
assert 20 not in {item for user, item, _, _ in base if user == 1}


472/131 204/61 2477732497/723744870 3.4234888559555525
[(20, 4), (33, 4), (61, 4), (117, 3), (155, 2), (160, 4), (171, 5), (189, 3), (202, 5), (265, 4)]


## Pitfall

About 3.423 against a hidden 4 is one residual. Averaging a few flattering residuals, or this one, does not replace the RMSE on all 9430. The row was chosen because it is the first line of `ua.test`, not because the error is small.

## Summary

User 1's base mean is $472/131$. *Angels and Insects* averages $204/61$ in the base file. The bias formula predicts $2477732497/723744870$, about 3.423, and the hidden rating is 4.
